# RNNs & LSTMs — Stock Price Trend Prediction

Sequential models that learn from temporal patterns.

In [ ]:
import micropip
await micropip.install(['scikit-learn','matplotlib','numpy'])
print('Ready!')

## Why RNNs?

Standard networks are memoryless — each input is processed independently.
RNNs maintain **hidden state** — memory of past inputs.

```
RNN:  h(t) = tanh(Wh · h(t-1) + Wx · x(t))
LSTM: adds 3 gates (forget, input, output) — solves vanishing gradient
GRU:  2 gates, fewer parameters, often as good as LSTM
```

In [ ]:
import numpy as np, matplotlib.pyplot as plt
np.random.seed(42)
t = np.arange(500)
price = 100 + 0.08*t + 15*np.sin(2*np.pi*t/52) + np.random.randn(500)*5
plt.figure(figsize=(12,4))
plt.plot(price,color='#6b21a8',lw=0.8)
plt.xlabel('Trading Days'); plt.ylabel('Price ($)')
plt.title('Simulated Stock: Trend + Seasonality + Noise')
plt.grid(alpha=0.3); plt.show()

## Sliding Window Features

In [ ]:
def make_sequences(series, window=20):
    X,y = [],[]
    for i in range(window,len(series)):
        X.append(series[i-window:i])
        y.append(series[i])
    return np.array(X), np.array(y)

window=20; X,y = make_sequences(price,window)
X_n=(X-X.mean())/X.std(); y_n=(y-y.mean())/y.std()
split=int(len(X)*0.8)
X_tr,X_te = X_n[:split], X_n[split:]
y_tr,y_te = y_n[:split], y_n[split:]
print(f"Each sample: {window} days of history → predict next day")
print(f"Train: {X_tr.shape}  |  Test: {X_te.shape}")

## Train & Evaluate

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_percentage_error
mlp = MLPRegressor(hidden_layer_sizes=(128,64),activation='relu',max_iter=1000,random_state=42)
mlp.fit(X_tr,y_tr)
y_pred = mlp.predict(X_te)
y_test_d = y_te*y.std()+y.mean(); y_pred_d = y_pred*y.std()+y.mean()
mape = mean_absolute_percentage_error(y_test_d,y_pred_d)*100
print(f"MAPE: {mape:.2f}%")
plt.figure(figsize=(12,4))
plt.plot(y_test_d[:100],label='Actual',color='#059669')
plt.plot(y_pred_d[:100],label='Predicted',color='#6b21a8',alpha=0.7)
plt.legend(); plt.title('Actual vs Predicted (first 100 test days)')
plt.grid(alpha=0.3); plt.show()

## Full LSTM (PyTorch — Kaggle GPU)

```python
import torch, torch.nn as nn

class StockLSTM(nn.Module):
    def __init__(self, hidden=64, layers=2):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, layers, batch_first=True, dropout=0.2)
        self.fc   = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)         # (batch, seq, hidden)
        return self.fc(out[:, -1, :]) # last timestep

model = StockLSTM()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()
```

## GRU vs LSTM

| | LSTM | GRU |
|---|---|---|
| Gates | 3 | 2 |
| Params | More | ~33% fewer |
| Speed | Slower | Faster |
| Performance | Often similar | Comparable |

**Rule:** Try GRU first. Use LSTM if sequences are very long (>200 steps).

---
**Your turn:** Change `window=5` and `window=60`. How does MAPE change with shorter vs longer history?